**Notebook Feito por:** MSc. Eng. Paulo de Souza Silva  
**Data:** Junho de 2026  
**Conteúdo retirado e adaptado de livros e artigos sobre Galerkin Descontínuo**  
**Agradecimentos:** Um agradecimento ao Prof. Dr. Alberto Nogueira pela disponibilização dos scripts em Python para DG 1D

# Aula 02 - Quadraturas

Antes de vermos como aplicar a ideia de Quadratura usando os polinômios de Jacobi, é interessante lembrar o conceito de Quadratura Gaussiana para uma base polinomial simples.

Existem diversos tipos de Quadratura Gaussiana, entre elas se destacam três estratégias:

* **Gauss-Legendre (GL):** É o padrão em precisão (integra exatamente polinômios de grau $2n-1$). No entanto, todos os nós de integração caem dentro do elemento (nenhum toca as fronteiras $\pm 1$).
* **Gauss-Lobatto-Legendre (GLL):** Sacrifica um pouco da precisão matemática (integra polinômios de grau $2n-3$), mas força a inclusão das fronteiras $\pm 1$ como pontos de quadratura.
* **Gauss-Radau:** Um meio-termo, onde apenas uma das fronteiras ($-1$ ou $1$) é forçada a ser um nó. É menos comum no DG nodal clássico, mas aparece em formulações upwind muito específicas.

Vamos fazer nossas deduções voltadas inicialmente para a estratégia GL.


## Quadratura de Gauss-Legendre

Na quadratura de Gauss-Legendre (GL) a nossa ideia é encontrar o valor exato da integral de uma função polinomial $f(x)$ utilizando um somatório.

Para isso, adotamos a estratégia de definir pesos ($w_i$) e raízes/zeros ($x_i$) que ponderam e avaliam a mesma função $f(x)$ de forma com que o resultado seja similar ao analítico da integral, matematicamente têm-se
\begin{equation}
    I = \int _a ^b f(x) dx \approx \sum_{i=1}^n w_i f(x_i)
\end{equation}
em que $n$ é a quantidade de pontos (raízes) que serão consideradas na aproximação.

No contexto do GL uma quantidade $n$ de pontos resolve a integral de forma exata para um polinômio de grau $2n-1$, ou seja, a nossa $f(x)$ pode ser escrita como:
\begin{equation}
    f(x) = \sum _{i=0}^{2n-1} c_i x^i
\end{equation}



### Construindo a solução para n = 2
#### Formulação da aproximação
Podemos começar analisando uma função que seja exata ao ser avaliada em **2 pontos (n=2)**. Por simplicidade,vamos tomar os limites de integração no intervalo $[a,b] = [-1,1]$. Logo
\begin{gather*}
    I = \int _{-1} ^1 f(x) dx = w_1 f(x_1) + w_2 f(x_2)
\end{gather*}

Como estamos considerando dois pontos o polinômio a ser adotado, para que a resposta seja exata, pode ser até de **grau 3** ($2\cdot \mathbf{2} -1$ = 3) ou seja
\begin{gather*}
    f(x) = c_0 + c_1 x + c_2 x^2 + c_3 x^3
\end{gather*}
  
  
Substituindo $f(x)$ na equação da integral temos:
\begin{align*}
    \int _{-1} ^1 (c_0 + c_1 x + c_2 x^2 + c_3 x^3) dx & =  w_1 (c_0 + c_1 x_1 + c_2 x_1^2 + c_3 x_1^3) \\ & +  w_2 (c_0 + c_1 x_2 + c_2 x_2^2 + c_3 x_2^3)
\end{align*}

#### Encontrando os pesos e as raízes
Vamos manipular a equação anterior a fim de isolar as constantes ($c_i$) e igualar a zero um dos lados:
\begin{align*}
0 &= c_0( w_1 + w_2 - \int _{-1} ^1 1dx) + \\
&+c_1( w_1x_1 + w_2x_2 - \int _{-1} ^1 xdx) +\\
&+c_2( w_1x_1^2 + w_2x_2^2 - \int _{-1} ^1 x^2dx)+\\
&+c_3( w_1x_1^3 + w_2x_2^3 - \int _{-1} ^1 x^3dx)
\end{align*}

Para termos uma fórmula polinomial os coeficientes $c_i$ **não podem ser nulos simultaneamente**, então cada um dos termos anteriores devem ser iguais a zero para que toda a equação resulte em zero, isto é:

**1º Termo**
$$( w_1 + w_2 - \int _{-1} ^1 1dx) = 0 ⟹ \int _{-1} ^1 1dx = w_1 + w_2$$
**2º Termo**
$$
( w_1x_1 + w_2x_2 - \int _{-1} ^1 xdx) = 0 ⟹ \int _{-1} ^1 xdx   = w_1x_1 + w_2x_2
$$
**3º Termo**
$$
( w_1x_1^2 + w_2x_2^2 - \int _{-1} ^1 x^2dx) = 0 ⟹ \int _{-1} ^1 x^2dx  = w_1x_1^2 + w_2x_2^2
$$
**4º Termo**
$$
( w_1x_1^3 + w_2x_2^3 - \int _{-1} ^1 x^3dx) = 0 ⟹ \int _{-1} ^1 x^3dx = w_1x_1^3 + w_2x_2^3
$$

Resolvendo as integrais
\begin{align*}
  w_1 + w_2 & = 2 \\
  w_1x_1 + w_2x_2 & = 0\\
  w_1x_1^2 + w_2x_2^2 & = \frac{2}{3} \\
  w_1x_1^3 + w_2x_2^3 & = 0
\end{align*}

Esse sistema apresenta então **4 equações** e **4 incógnitas** ($w_1, w_2, x_1$ e $x_2$), tendo solução única. Note que o mesmo é não linear o que já dificulta o processo de resolução.

Utilizando a biblioteca Simbólica do Python (ver célula seguinte) chegamos nas soluções:
\begin{gather*}
  w_1 = 1 \hspace{1cm} w_2 = 1 \hspace{1cm} x_1 = -\frac{\sqrt{3}}{3} \hspace{1cm} x_2 = \frac{\sqrt{3}}{3}
\end{gather*}

Logo a nossa integral considerando **dois pontos** terá **solução exata** para um polinômio **até grau 3** se calculada como:
\begin{gather*}
    I = \int _{-1} ^1 f(x) dx = 1 \times f\left(-\frac{\sqrt{3}}{3}\right) + 1 \times f\left(\frac{\sqrt{3}}{3}\right)
\end{gather*}


In [ ]:
import sympy as sp

w1,w2,x1,x2 = sp.symbols("w1 w2 x1 x2")

eqs = [
    sp.Eq(w1+w2,2),
    sp.Eq(w1*x1+w2*x2,0),
    sp.Eq(w1*x1**2+w2*x2**2,sp.Rational(2,3)),
    sp.Eq(w1*x1**3+w2*x2**3,0)
]

sol = sp.solve(eqs,[w1,w2,x1,x2], dict=True)

print(sol)

[{w1: 1, w2: 1, x1: -sqrt(3)/3, x2: sqrt(3)/3}, {w1: 1, w2: 1, x1: sqrt(3)/3, x2: -sqrt(3)/3}]


### A Transformação de Domínio: Do Elemento Físico para o Padrão

Até agora, deduzimos os pesos e as raízes da quadratura estritamente para o intervalo $[-1, 1]$. No entanto, em problemas reais, precisamos calcular integrais em domínios arbitrários $x \in [a, b]$.

> **A Filosofia do DG:** Na formulação Galerkin Descontínuo, o domínio global do problema é dividido em dezenas ou centenas de "elementos físicos" de tamanhos variados. Para não termos que recalcular raízes e pesos para cada elemento da malha, a estratégia é mapear todos eles para um único **Elemento Padrão** paramétrico, onde $\xi \in [-1, 1]$.

Para garantir a equivalência entre a integral no mundo físico e no elemento padrão, aplicamos uma transformação linear (afim) de coordenadas:
\begin{equation*}
    x = m \xi + c \implies dx = m \, d\xi
\end{equation*}

Queremos que a fronteira $\xi = -1$ mapeie exatamente para $x = a$, e que $\xi = 1$ mapeie para $x = b$. Resolvendo esse sistema, encontramos os coeficientes da reta:
\begin{gather*}
    m = \frac{b-a}{2} \hspace{1cm} c = \frac{b+a}{2}
\end{gather*}

Substituindo essas relações, a integral sobre o domínio físico é reescrita em função do nosso domínio padrão paramétrico:
\begin{gather*}
    \int _a ^b f(x) dx = \int _{-1}^1 f\left( \frac{b-a}{2} \xi + \frac{b+a}{2} \right) \frac{b-a}{2} d\xi
\end{gather*}

> **O Jacobiano:** Note que o termo $m = \frac{b-a}{2}$ "saiu" do diferencial $dx$. Na matemática do DG, a derivada espacial $\frac{dx}{d\xi}$ é o **Jacobiano da transformação ($J$)** para o caso 1D. Ele é o "fator de escala" que mede o quanto o elemento padrão precisou ser esticado ou encolhido para virar o elemento físico.

Para deixar a notação mais limpa, podemos definir $F(\xi) = f(m\xi + c)$, que é a nossa função original avaliada já nos pontos mapeados. Assim, a aplicação da quadratura numérica generalizada ganha a sua forma final:
\begin{gather*}
    \int _a ^b f(x) dx = \frac{b-a}{2} \int _{-1}^1 F(\xi) d\xi \approx \frac{b-a}{2} \sum_{i=1}^n w_i F(\xi_i)
\end{gather*}

**Consideração Numérica (Boas Práticas):** Como as raízes ($\xi_i$) e os pesos ($w_i$) são frequentemente números irracionais (ex: $\pm 1/\sqrt{3}$), o código deve sempre calcular e armazenar essas variáveis utilizando a máxima precisão de ponto flutuante disponível na linguagem (`float64`), evitando arredondamentos que destruiriam a convergência espectral do método.

### Codando a Teoria do GL

Acabamos de provar analiticamente que, para $n=2$ pontos, os pesos são $w_1 = w_2 = 1$ e as raízes no domínio padrão $[-1, 1]$ são $\xi = \pm 1/\sqrt{3}$. A matemática nos garante que essa quadratura com 2 pontos integrará perfeitamente polinômios de grau até $2n-1 = 3$.

Vamos construir uma função simples em Python para aplicar essa lógica e a transformação de domínio que deduzimos. Para enriquecer nosso teste, adicionaremos também os valores tabelados para $n=3$ pontos (que deve integrar perfeitamente até o grau 5).

Valores para n = 3 retirados de [Legendre-Gauss Quadrature](https://mathworld.wolfram.com/Legendre-GaussQuadrature.html)

In [ ]:
import math
from typing import Callable

def int_quad_gauss(f: Callable[[float], float], a: float, b: float, n: int = 2) -> float:
    """
    Calcula a integral da função f(x) no intervalo [a,b] usando Quadratura Gaussiana.
    Função com fins didáticos: suporta n=2 ou n=3 pontos de integração.
    """
    # Transformação de domínio de [a, b] para [-1, 1]
    m = (b - a) / 2.0
    c = (b + a) / 2.0

    if n == 2:
        # Raízes e Pesos para n=2 (Exato para polinômios até grau 3)
        xi = [-1.0 / math.sqrt(3.0), 1.0 / math.sqrt(3.0)]
        w = [1.0, 1.0]
    elif n == 3:
        # Raízes e Pesos para n=3 (Exato para polinômios até grau 5)
        xi = [-math.sqrt(3.0 / 5.0), 0.0, math.sqrt(3.0 / 5.0)]
        w = [5.0 / 9.0, 8.0 / 9.0, 5.0 / 9.0]
    else:
        raise ValueError("Para nossa ilustração, escolha n=2 ou n=3.")

    integral = 0.0
    for i in range(n):
        # Avaliando a função no ponto transformado e multiplicando pelo peso
        integral += w[i] * f(m * xi[i] + c)

    # Multiplicamos pelo jacobiano da transformação 1D (m) no final
    return m * integral

### Grau de Exatidão

Sabemos que a integral analítica de $f(x) = x^3$ no intervalo $[0, 2]$ é:
$$\int_{0}^{2} x^3 dx = \left[ \frac{x^4}{4} \right]_0^2 = \frac{16}{4} - 0 = 4.0$$

Como $x^3$ é grau 3, nossa quadratura com $n=2$ **deve** cravar o resultado exato.  

Mas o que acontece se tentarmos integrar $f(x) = x^4$ com os mesmos $n=2$ pontos?

A integral exata é $\frac{32}{5} = 6.4$. Será que o código quebra?

In [ ]:
# Definindo as funções para teste
def func_grau_3(x: float) -> float:
    return x**3

def func_grau_4(x: float) -> float:
    return x**4

print("=== Testando f(x) = x^3 ===")
res_2pts = int_quad_gauss(func_grau_3, 0.0, 2.0, n=2)
print(f"Resultado analítico: 4.0")
print(f"Quadratura (n=2):    {res_2pts:.5f}  <-- Exato!")

print("\n=== Testando f(x) = x^4 ===")
res_2pts_fail = int_quad_gauss(func_grau_4, 0.0, 2.0, n=2)
res_3pts = int_quad_gauss(func_grau_4, 0.0, 2.0, n=3)

print(f"Resultado analítico: 6.4")
print(f"Quadratura (n=2):    {res_2pts_fail:.5f}  <-- Erro! (Faltou precisão)")
print(f"Quadratura (n=3):    {res_3pts:.5f}  <-- Exato novamente!")

=== Testando f(x) = x^3 ===
Resultado analítico: 4.0
Quadratura (n=2):    4.00000  <-- Exato!

=== Testando f(x) = x^4 ===
Resultado analítico: 6.4
Quadratura (n=2):    6.22222  <-- Erro! (Faltou precisão)
Quadratura (n=3):    6.40000  <-- Exato novamente!


## Dilema das Fronteiras no DG (Legendre vs. Lobatto)

A Quadratura de Gauss-Legendre (GL) com $n$ pontos integra perfeitamente polinômios de grau $2n-1$. No entanto, ela possui uma característica peculiar: **todos os nós de integração caem estritamente no interior do elemento** (nenhum ponto toca as fronteiras $-1$ ou $1$).

Para o Método Galerkin Descontínuo (DG), isso gera um desafio prático. No DG, os elementos da malha são independentes e só "conversam" entre si através dos **fluxos numéricos nas interfaces (fronteiras)**.
* Se usarmos a malha de Gauss-Legendre, não temos a solução avaliada exatamente na fronteira. Teremos que gastar processamento para **interpolar** a solução do interior do elemento para as bordas a cada passo de tempo.

Para contornar isso, os pesquisadores frequentemente adotam a quadratura de **Gauss-Lobatto**. A regra de Lobatto **força a inclusão das fronteiras ($-1$ e $1$) como pontos de integração**.

* **A Vantagem:** O valor da solução na fronteira já está calculado "de graça". Não precisamos de interpolação para calcular os fluxos entre os elementos.
* **O Preço:** Ao engessar dois pontos nas fronteiras, perdemos graus de liberdade na otimização matemática. A quadratura de Lobatto com $n$ pontos só integra exatamente polinômios de grau $2n-3$ (duas ordens de exatidão a menos que a GL tradicional).

*(Nota: Existe também a quadratura de Gauss-Radau, que fixa apenas UMA das fronteiras, muito utilizada em formulações puramente upwind, mas o embate principal no DG nodal é sempre entre GL e GLL).*

### Quadratura de Gauss-Lobatto-Legendre (GLL)

Para construir a quadratura de Gauss-Lobatto com $n$ pontos no intervalo $[-1, 1]$, nós **forçamos** a existência de raízes nas fronteiras: $\xi_1 = -1$ e $\xi_n = 1$. Portanto, nossa aproximação geral passa a ser:

\begin{gather*}
    I = \int _{-1} ^1 f(x) dx \approx w_1 f(-1) + w_n f(1) + \sum_{i = 2} ^{n-1} w_i f(x_i)
\end{gather*}

**Exemplo para $n=3$ pontos:**  
Para garantir a exatidão, precisamos lembrar que a GLL integra perfeitamente polinômios de até grau $2n-3$. Logo, com $n=3$, exigimos exatidão para polinômios de grau até $3$ ($2 \times 3 - 3 = 3$)
\begin{gather*}
    f(x) = c_0 + c_1 x + c_2 x^2 + c_3 x^3
\end{gather*}

Vamos substituir esse polinômio na nossa aproximação de GLL (já avaliando as raízes extremas $x_1 = -1$ e $x_3 = 1$ e distribuindo os respectivos pesos):
\begin{align*}
    \int _{-1} ^1 (c_0 + c_1 x + c_2 x^2 + c_3 x^3) dx &= w_1 (c_0 - c_1 + c_2 - c_3 ) \\
    &+ w_2 (c_0 + c_1 x_2 + c_2 x_2^2 + c_3 x_2^3) \\
    &+ w_3 (c_0 + c_1 + c_2 + c_3)
\end{align*}

Agrupando os termos que possuem os mesmos coeficientes $c_i$ em comum e passando a integral para o outro lado, ficamos com a seguinte relação:
\begin{align*}
0 &= c_0\left( w_1 + w_2 + w_3 - \int _{-1} ^1 1dx \right) + \\
  &+ c_1\left( -w_1 + w_2x_2 + w_3 - \int _{-1} ^1 xdx \right) + \\
  &+ c_2\left( w_1 + w_2x_2^2 + w_3 - \int _{-1} ^1 x^2dx \right) + \\
  &+ c_3\left( -w_1 + w_2x_2^3 + w_3 - \int _{-1} ^1 x^3dx \right)
\end{align*}

Para que a integração numérica seja exata para qualquer combinação possível do polinômio, os coeficientes $c_i$ **não podem ser nulos simultaneamente**. Logo, a única forma da equação inteira resultar sempre em zero é se cada componente for individualmente igual a zero.

**1º Termo (Grau 0):**
$$\left( w_1 + w_2 + w_3 - \int _{-1} ^1 1dx \right) = 0 \implies \int _{-1} ^1 1dx = w_1 + w_2 + w_3$$

**2º Termo (Grau 1):**
$$\left( -w_1 + w_2x_2 + w_3 - \int _{-1} ^1 xdx \right) = 0 \implies \int _{-1} ^1 xdx = -w_1 + w_2x_2 + w_3$$

**3º Termo (Grau 2):**
$$\left( w_1 + w_2x_2^2 + w_3 - \int _{-1} ^1 x^2dx \right) = 0 \implies \int _{-1} ^1 x^2dx = w_1 + w_2x_2^2 + w_3$$

**4º Termo (Grau 3):**
$$\left( -w_1 + w_2x_2^3 + w_3 - \int _{-1} ^1 x^3dx \right) = 0 \implies \int _{-1} ^1 x^3dx = -w_1 + w_2x_2^3 + w_3$$

Resolvendo as integrais analíticas do lado esquerdo, montamos o nosso sistema não-linear final:
\begin{align*}
  w_1 + w_2 + w_3 &= 2 \\
 -w_1 + w_2x_2 + w_3 &= 0 \\
  w_1 + w_2x_2^2 + w_3 &= \frac{2}{3} \\
 -w_1 + w_2x_2^3 + w_3 &= 0
\end{align*}

Desse contexto, temos então **4 equações** e **4 incógnitas** acopladas ($w_1, w_2, w_3$ e $x_2$). Resolvendo o sistema, chegamos à solução exata para a GLL com 3 pontos:
\begin{gather*}
  w_1 = \frac{1}{3} \hspace{1cm} w_2 = \frac{4}{3} \hspace{1cm} w_3 = \frac{1}{3} \hspace{1cm} x_2 = 0
\end{gather*}

> Os valores dessas incógnitas não são triviais de isolar no papel e foram obtidos utilizando a biblioteca de **Python Simbólico (SymPy)**.

In [ ]:
import sympy as sp

w1,w2,w3,x2 = sp.symbols("w1 w2 w3 x2")

eqs = [
    sp.Eq(w1+w2+w3,2),
    sp.Eq(-w1+w2*x2+w3,0),
    sp.Eq(w1+w2*x2**2+w3,sp.Rational(2,3)),
    sp.Eq(-w1+w2*x2**3+w3,0)
]

sol = sp.solve(eqs,[w1,w2,w3,x2], dict=True)

print(sol)

[{w1: 1/3, w2: 4/3, w3: 1/3, x2: 0}]


### Codando a Teoria do GLL

Na ocasião definiremos somente o GLL para n=3.

> **Observação** Para o caso da GLL também devemos generalizar a questão dos extremos de integração. Como 0 processo de mudança de variável é similar ao apresentado no caso do GL ele não será repetido aqui.

In [ ]:
def int_quad_lobatto(f: Callable[[float], float], a: float, b: float) -> float:
    """
    Calcula a integral usando Quadratura Gauss-Lobatto-Legendre para n=3 pontos.
    Exato para polinômios até grau 2n-3 = 3.
    """
    m = (b - a) / 2.0
    c = (b + a) / 2.0

    # Raízes e Pesos GLL para n=3
    xi = [-1.0, 0.0, 1.0]
    w = [1.0 / 3.0, 4.0 / 3.0, 1.0 / 3.0]

    integral = 0.0
    for i in range(3):
        integral += w[i] * f(m * xi[i] + c)

    return m * integral

### Teste da Integração de Gauss-Lobatto

Usando a função $x^3$ calculada no intervalo de $[0, 2]$, como feito anteriormente, sabemos que o resultado analítico é $4.0$.

In [ ]:
print("=== Testando f(x) = x^3 ===")
res_2pts = int_quad_lobatto(func_grau_3, 0.0, 2.0)
print(f"Resultado analítico: 4.0")
print(f"Quadratura GLL (n=3):  {res_2pts:.3f}  <-- Exato!")

=== Testando f(x) = x^3 ===
Resultado analítico: 4.0
Quadratura GLL (n=3):  4.000  <-- Exato!


### Teste de Exatidão

Devido ao seu formalismo o GLL "paga um preço" das Fronteiras. Lembre-se:
* Gauss-Legendre com $n=3$ integra exatamente até o grau $2n-1 = 5$.
* Gauss-Lobatto com $n=3$ integra exatamente até o grau $2n-3 = 3$.

Vamos usar a mesma função $f(x) = x^4$ no intervalo $[0, 2]$, cuja integral analítica exata é $6.4$.

O que acontece se compararmos as duas formulações com o mesmo número de pontos ($n=3$)?

In [ ]:
print("=== Integrando f(x) = x^4 no intervalo [0, 2] ===\n")
print("Resultado analítico: 6.4")

# Usando o GL (n=3) construído anteriormente
res_gl_3 = int_quad_gauss(func_grau_4, 0.0, 2.0, n=3)
print(f"Gauss-Legendre (GL) n=3:  {res_gl_3:.5f}  <-- Exato (Suporta até grau 5)")

# Usando o GLL (n=3)
res_gll_3 = int_quad_lobatto(func_grau_4, 0.0, 2.0)
print(f"Gauss-Lobatto (GLL) n=3:  {res_gll_3:.5f}  <-- Erro! (Só suporta até grau 3)")

=== Integrando f(x) = x^4 no intervalo [0, 2] ===

Resultado analítico: 6.4
Gauss-Legendre (GL) n=3:  6.40000  <-- Exato (Suporta até grau 5)
Gauss-Lobatto (GLL) n=3:  6.66667  <-- Erro! (Só suporta até grau 3)


## Complexidade via Cálculo Manual (Próximos Passos)

Neste ponto, a mensagem deve estar clara: conseguimos deduzir as raízes e os pesos para $n=2$ e $n=3$ na "força bruta" algébrica, montando e resolvendo sistemas de equações não-lineares. Mas repare na **trabalheira** que isso deu tanto para a Quadratura de Gauss-Legendre quanto para a de Gauss-Lobatto!

No Método Galerkin Descontínuo (DG), o grande trunfo é a convergência $p$ (aumentar o grau do polinômio para ganhar precisão). O que acontece se precisarmos rodar uma simulação com aproximação de grau $P=10$ ou $P=15$?

* **A Matemática Simbólica:** Montar e tentar resolver um sistema acoplado com dezenas de variáveis para achar raízes e pesos no papel (ou até mesmo em softwares de álgebra) torna-se um pesadelo impraticável.
* **Tabelas não Escalam:** Ficar buscando tabelas na literatura e "chumbando" (hardcoding) dezenas de números com 16 casas decimais no código é ineficiente e um convite aberto para erros de digitação que destruiriam a estabilidade do solver.

### Abordagem Algorítmica

Não podemos depender de tabelas. Precisamos que o nosso código "fabrique" suas próprias raízes e pesos dinamicamente, para qualquer ordem, no exato momento em que a simulação começar.

Para a nossa sorte, não precisaremos deduzir sistemas gigantescos. Na **Aula 03**, nós vamos resgatar os nossos **Polinômios de Jacobi** construídos na Aula 01 e conectá-los com ferramentas puras de cálculo numérico.

O trabalho manual termina aqui!

## Extras e Desafios para a Próxima Semana

Para não perdermos o ritmo até o nosso próximo encontro, deixo aqui materiais de aprofundamento e dois desafios práticos baseados no que construímos hoje.

### 📖 Recomendação de Leitura
Leiam as formulações gerais das quadraturas no site do *Wolfram MathWorld*. Observem atentamente as expressões fechadas para os pesos: são exatamente essas equações matemáticas que transformaremos em código Python na Aula 03.
* [Legendre-Gauss Quadrature (Wolfram)](https://mathworld.wolfram.com/Legendre-GaussQuadrature.html)
* [Lobatto Quadrature (Wolfram)](https://mathworld.wolfram.com/LobattoQuadrature.html)

### 💻 Desafios Práticos (Mão na Massa)

**Desafio 1: O Teste de Estresse**  
Usando as funções `int_quad_gauss` e `int_quad_lobatto` do notebook de hoje, saiam do terreno seguro dos polinômios de baixo grau:
* Tentem integrar um polinômio de grau 6 usando a função de $n=3$ pontos.
* O que acontece se tentarem integrar funções não-polinomiais, como $f(x) = \sin(x)$ ou $f(x) = e^x$? A quadratura funciona?
* Mudem o intervalo de integração $[a, b]$ para limites gigantescos (ex: $x \in [0, 100]$) e observem como o "esticamento" do Jacobiano afeta o erro de aproximação.

> **💡 Dica Teórica para o Desafio:** Por que a quadratura consegue aproximar funções como $e^x$ ou $\sin(x)$ se a matemática foi deduzida puramente para polinômios? Lembre-se das **Séries de Taylor**! Funções transcendentais podem ser escritas como um polinômio infinito ($e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \dots$). Uma quadratura que exija exatidão de grau $2n-1$ integrará perfeitamente os primeiros $2n-1$ termos dessa série. O erro residual que você verá no código é simplesmente o peso dos termos de ordem superior que ficaram de fora!

**Desafio 2: Aquecimento para a Aula 03**  
Copiem as funções `jacobi_p` e `djacobi_p` (Aula 01) para este notebook. Após lerem a teoria nos links do Wolfram, tentem utilizar nossos polinômios espectrais para calcular na "força bruta" os pesos de uma quadratura com $n=4$.
* *Dica:* Se você conseguir fazer o polinômio da Aula 01 alimentar a integral da Aula 02, você já descobriu 50% do algoritmo que vamos programar no próximo encontro!
